## Creating a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits

In [2]:
import os
import json
from dotenv import load_dotenv


In [3]:
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

In [ ]:
if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    

In [5]:
from openai import OpenAI

openai = OpenAI()

In [ ]:
#fetch links from the website - https://huggingface.co"
from scraper import fetch_website_links

link = "https://www.anthropic.com/"

links = fetch_website_links(link)

links

### We get all the links from the website, these links includes business links and some links which are not relevent to the website 


### We need to Make sure, we get those links that are relevent to create a brochure using AI

## Let's do it

## First Step - Have GPT-5-nano figure out Which Links are relevant

In [7]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [8]:
def get_links_user_prompt(url):
    user_prompt = f"""
    Here is the list of links on the website {url} -
    Please decide which of these are relevant web links for a brochure about the company, 
    respond with the full https URL in JSON format.
    Do not include Terms of Service, Privacy, email links.

    Links (some might be relative links):

    """
    links = fetch_website_links(link)
    user_prompt += "\n".join(links)
    return user_prompt

In [ ]:
print(get_links_user_prompt("https://www.anthropic.com/"))

In [10]:
MODEL = "gpt-5-nano"

In [11]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create( model=MODEL, messages=[
        { "role":"system", "content": link_system_prompt},
        {"role":"user", "content": get_links_user_prompt(url)
        }],
        response_format={"type" : "json_object"}
        )
    results = response.choices[0].message.content
    links = json.loads(results)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [ ]:
select_relevant_links("https://www.anthropic.com/")

## Second Step - Let's Make the Brochure

In [12]:
from scraper import fetch_website_contents


def make_brochure(url):
    contents = fetch_website_contents(url)
    links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in links["links"]:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [ ]:
print(make_brochure("https://www.anthropic.com/"))

In [13]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [14]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += make_brochure(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [15]:
from IPython.display import Markdown, display

def create_brochure(company_name, url):
    response = openai.chat.completions.create(model=Model, messages=[
        {"role" : "system", "content":brochure_system_prompt },
        {"role":"user", "content" : get_brochure_user_prompt(company_name, url)},
        ])
    result = response.choices[0].message.content
    display(Markdown(result))

In [ ]:
create_brochure("Antropic", "https://www.anthropic.com/")

## Lets improvise it with Streaming - instead of waiting for the full response, you display it word by word like ChatGPT does

In [ ]:

from IPython.display import update_display

def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(model = MODEL, messages=[
        {"role" : "system", "content":brochure_system_prompt },
        {"role":"user", "content" : get_brochure_user_prompt(company_name, url)},],
        stream=True
    )
    response = ""

    # Creates an empty placeholder in the notebook output. display_id=True gives it a unique ID so you can update it later.
    display_handle = display(Markdown(""), display_id=True)

    for chunk in stream:

        # Each chunk contains a small piece of text (could be one word or few characters). .delta means "the new piece
        response += chunk.choices[0].delta.content or ''

        # Updates that same placeholder with the growing response on every chunk
        update_display(Markdown(response), display_id=display_handle.display_id)



In [ ]:
stream_brochure("Antropic", "https://www.anthropic.com/")

## Gradio for Brochure

In [ ]:
import gradio as gr

In [19]:
def display_brochure(company_name, url):
    messages = [
        {"role":"system", "content" : brochure_system_prompt},
        {"role":"user", "content" : get_brochure_user_prompt(company_name, url)}
    ]

    response = openai.chat.completions.create(model = "gpt-5-nano", messages=messages, stream = True)
    result = ""
    for chunk in response:
        result += chunk.choices[0].delta.content or ""
        yield result

In [ ]:
message_Input = gr.Textbox(label = "Enter the Company Name")
message_Input2 = gr.Textbox(label = "Enter the Company Website URL")
message_output = gr.Markdown(label = "Response")

gr.Interface(
    fn = display_brochure,
    title = "Brochure Generate using GPT",
    inputs = [message_Input, message_Input2],
    outputs = [message_output],
    flagging_mode = "never"
).launch()
